# Day 5: Activation Functions - MNIST Digit Classification

This notebook demonstrates the impact of different activation functions on neural network performance using the MNIST digit classification task.

**Objective:** Compare how different activation functions (ReLU, Sigmoid, Tanh, Leaky ReLU) affect training accuracy and convergence speed.

**Key Learning Points:**
- Activation functions introduce non-linearity to neural networks
- Different activation functions can significantly impact model performance
- ReLU is the modern standard for hidden layers
- Sigmoid and Tanh suffer from vanishing gradient problems in deep networks


In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")


## Step 1: Load and Prepare MNIST Dataset

We'll use the standard MNIST dataset with 60,000 training images and 10,000 test images of handwritten digits (0-9).


In [ ]:
# Define data transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load training and test datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Number of classes: {len(train_dataset.classes)}")


## Step 2: Define Neural Network with Configurable Activation Function

We'll create a neural network class that allows us to easily switch between different activation functions while keeping all other parameters the same.


In [ ]:
class MNISTNet(nn.Module):
    """
    Neural network for MNIST classification with configurable activation function.
    
    Architecture:
    - Input: 28x28 = 784 pixels
    - Hidden Layer 1: 128 neurons
    - Hidden Layer 2: 64 neurons
    - Output: 10 classes (digits 0-9)
    """
    def __init__(self, activation='relu'):
        super(MNISTNet, self).__init__()
        
        # Flatten 28x28 image to 784
        self.flatten = nn.Flatten()
        
        # Define layers
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        
        # Select activation function
        if activation.lower() == 'relu':
            self.activation = nn.ReLU()
        elif activation.lower() == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation.lower() == 'tanh':
            self.activation = nn.Tanh()
        elif activation.lower() == 'leaky_relu':
            self.activation = nn.LeakyReLU(negative_slope=0.01)
        else:
            raise ValueError(f"Unknown activation function: {activation}. Choose from: relu, sigmoid, tanh, leaky_relu")
        
        self.activation_name = activation.upper()
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)  # No activation on output layer (we'll use CrossEntropyLoss)
        return x

# Test the model
model = MNISTNet(activation='relu')
print(f"Model architecture with {model.activation_name}:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")


## Step 3: Training Function

This function trains the model and returns training history for comparison.


In [ ]:
def train_model(activation='relu', num_epochs=5, learning_rate=0.001):
    """
    Train a model with specified activation function.
    
    Parameters:
    - activation: 'relu', 'sigmoid', 'tanh', or 'leaky_relu'
    - num_epochs: Number of training epochs
    - learning_rate: Learning rate for optimizer
    
    Returns:
    - model: Trained model
    - history: Dictionary with training history
    """
    # Create model
    model = MNISTNet(activation=activation).to(device)
    
    # Loss and optimizer (same for all models)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_acc': []
    }
    
    print(f"\n{'='*60}")
    print(f"Training with {model.activation_name} activation function")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Statistics
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
        
        train_accuracy = 100 * correct_train / total_train
        avg_train_loss = train_loss / len(train_loader)
        
        # Test phase
        model.eval()
        correct_test = 0
        total_test = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total_test += labels.size(0)
                correct_test += (predicted == labels).sum().item()
        
        test_accuracy = 100 * correct_test / total_test
        
        # Store history
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_accuracy)
        history['test_acc'].append(test_accuracy)
        
        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Train Acc: {train_accuracy:.2f}%, "
              f"Test Acc: {test_accuracy:.2f}%")
    
    print(f"\nFinal Test Accuracy with {model.activation_name}: {test_accuracy:.2f}%")
    
    return model, history


## Step 4: Compare Different Activation Functions

**Change the activation function below to see the difference!**

Available options:
- `'relu'` - Rectified Linear Unit (modern standard)
- `'sigmoid'` - Sigmoid function (legacy, suffers from vanishing gradients)
- `'tanh'` - Hyperbolic Tangent (better than sigmoid but still has issues)
- `'leaky_relu'` - Leaky ReLU (solves dying ReLU problem)


In [ ]:
# ============================================
# CHANGE THIS TO TEST DIFFERENT ACTIVATION FUNCTIONS
# ============================================
ACTIVATION_FUNCTION = 'relu'  # Options: 'relu', 'sigmoid', 'tanh', 'leaky_relu'
# ============================================

# Training parameters (same for all models)
NUM_EPOCHS = 5
LEARNING_RATE = 0.001

# Train the model
model, history = train_model(
    activation=ACTIVATION_FUNCTION,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE
)


## Step 5: Visualize Training Progress


In [ ]:
# Plot training history
plt.figure(figsize=(15, 5))

# Plot 1: Training Loss
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], 'b-', linewidth=2, label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title(f'Training Loss ({ACTIVATION_FUNCTION.upper()})')
plt.grid(True, alpha=0.3)
plt.legend()

# Plot 2: Training Accuracy
plt.subplot(1, 3, 2)
plt.plot(history['train_acc'], 'g-', linewidth=2, label='Train Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title(f'Training Accuracy ({ACTIVATION_FUNCTION.upper()})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim([0, 100])

# Plot 3: Test Accuracy
plt.subplot(1, 3, 3)
plt.plot(history['test_acc'], 'r-', linewidth=2, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title(f'Test Accuracy ({ACTIVATION_FUNCTION.upper()})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim([0, 100])

plt.tight_layout()
plt.show()

print(f"\nSummary for {ACTIVATION_FUNCTION.upper()}:")
print(f"  Final Training Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"  Final Test Accuracy: {history['test_acc'][-1]:.2f}%")
print(f"  Final Training Loss: {history['train_loss'][-1]:.4f}")


## Step 6: Compare All Activation Functions (Optional)

Run this cell to compare all activation functions side-by-side. This will take longer as it trains 4 models.


In [ ]:
# Uncomment the code below to compare all activation functions
# This will train 4 models and compare their performance

"""
# List of activation functions to compare
activations = ['relu', 'sigmoid', 'tanh', 'leaky_relu']

# Store results for all activations
all_results = {}

print("="*60)
print("COMPARING ALL ACTIVATION FUNCTIONS")
print("="*60)

for act in activations:
    print(f"\n{'='*60}")
    model, history = train_model(activation=act, num_epochs=5, learning_rate=0.001)
    all_results[act] = history
    print(f"{'='*60}\n")

# Plot comparison
plt.figure(figsize=(15, 5))

# Plot 1: Training Loss Comparison
plt.subplot(1, 3, 1)
for act, hist in all_results.items():
    plt.plot(hist['train_loss'], linewidth=2, label=act.upper())
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Training Accuracy Comparison
plt.subplot(1, 3, 2)
for act, hist in all_results.items():
    plt.plot(hist['train_acc'], linewidth=2, label=act.upper())
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim([0, 100])

# Plot 3: Test Accuracy Comparison
plt.subplot(1, 3, 3)
for act, hist in all_results.items():
    plt.plot(hist['test_acc'], linewidth=2, label=act.upper())
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Test Accuracy Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim([0, 100])

plt.tight_layout()
plt.show()

# Print summary table
print("\n" + "="*60)
print("FINAL RESULTS COMPARISON")
print("="*60)
print(f"{'Activation':<15} {'Train Acc':<15} {'Test Acc':<15} {'Final Loss':<15}")
print("-"*60)
for act, hist in all_results.items():
    print(f"{act.upper():<15} {hist['train_acc'][-1]:>10.2f}%    {hist['test_acc'][-1]:>10.2f}%    {hist['train_loss'][-1]:>10.4f}")
print("="*60)
"""


## Step 7: Visualize Some Predictions

Let's see how well our model performs on some test images.


In [ ]:
# Get a batch of test images
model.eval()
dataiter = iter(test_loader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)

# Move to CPU for visualization
images = images.cpu()
labels = labels.cpu()
predicted = predicted.cpu()

# Visualize predictions
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.ravel()

for i in range(16):
    axes[i].imshow(images[i].squeeze(), cmap='gray')
    axes[i].axis('off')
    
    # Color code: green for correct, red for incorrect
    color = 'green' if predicted[i] == labels[i] else 'red'
    axes[i].set_title(f'True: {labels[i].item()}\nPred: {predicted[i].item()}', 
                      color=color, fontsize=10)

plt.suptitle(f'Predictions with {ACTIVATION_FUNCTION.upper()} Activation', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate accuracy on this batch
correct = (predicted == labels).sum().item()
total = labels.size(0)
print(f"Accuracy on this batch: {100 * correct / total:.2f}% ({correct}/{total})")


## Key Observations

**Expected Results:**

1. **ReLU**: Should achieve the highest accuracy (~97-98%) and fastest convergence
   - No vanishing gradient problem
   - Computationally efficient
   - Modern standard for hidden layers

2. **Leaky ReLU**: Similar to ReLU, slightly better in some cases
   - Solves the "dying ReLU" problem
   - May perform slightly better than ReLU

3. **Tanh**: Lower accuracy than ReLU (~95-96%)
   - Zero-centered (better than sigmoid)
   - Still suffers from vanishing gradients in deep networks

4. **Sigmoid**: Lowest accuracy (~90-93%)
   - Not zero-centered
   - Severe vanishing gradient problem
   - Slow convergence

**Try changing the `ACTIVATION_FUNCTION` variable in Step 4 to see the differences!**
